# Huni — ground-truth extractor

**Upload an original MP3, get back the ground-truth JSON the app uses.**

Runs: Demucs (isolate the vocal) → CREPE (pitch) → note segmentation →
Whisper (word timings) → merged into `{syllable, start, end, note}`.

### Before you start
1. **Runtime → Change runtime type → T4 GPU.** On CPU this takes hours.
2. Upload the **original recording**, not a karaoke/minus-one version —
   the whole method depends on extracting a vocal, and a minus-one has none.

Whisper and CREPE run offline here, so this uses `large-v3` rather than the
`tiny` model the phone is limited to. Accuracy matters more than speed when
the output is a reference other people get scored against.

In [ ]:
#@title 1 — Install (a few minutes)
# crepe's setup.py imports pkg_resources, removed in setuptools >= 81, so
# it must build against a pinned setuptools with build isolation off.
!pip install -q "setuptools<81" wheel
!pip install -q --no-build-isolation crepe
!pip install -q demucs openai-whisper soundfile resampy librosa

import os, re, json, glob, subprocess
import numpy as np, librosa, whisper
import tensorflow as tf

# crepe is checked in cell 3b, which falls back to torchcrepe if it can't
# build its model here - so a crepe problem doesn't stop the run.
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE — set Runtime > T4 GPU')


In [ ]:
#@title 2 — Upload your MP3(s)
import os, re
from google.colab import files

print('Select one or more ORIGINAL mp3 files (not karaoke versions)...')
uploaded = files.upload()
paths = list(uploaded)

def clean_title(filename):
    """Download names carry artist and video tags; the app keys on the song
    title alone, and the JSON filename has to match it.
    'Adie - Tahanan (Official Lyric Video).mp3' -> 'Tahanan'"""
    t = os.path.splitext(filename)[0]
    t = re.sub(r'\((?:official|lyric|music|video|audio|hd|mv)[^)]*\)', '', t, flags=re.I)
    t = re.sub(r'\[[^\]]*\]', '', t)
    t = re.sub(r'\(\d+\)', '', t)          # (1), (2) from repeat downloads
    if ' - ' in t:
        t = t.split(' - ')[-1]              # 'Artist - Title' -> 'Title'
    return re.sub(r'\s+', ' ', t).strip()

TITLES = {p: clean_title(p) for p in paths}
for p in paths:
    print(f'  {os.path.getsize(p)/1e6:5.1f} MB  {p}')
    print(f'            -> title: "{TITLES[p]}"')

print()
print('If a title is wrong, correct it before cell 4, e.g.:')
print('   TITLES[paths[0]] = "Tahanan"')


In [ ]:
#@title 3 — Settings
LANGUAGE        = 'tl'       #@param ['tl','en','auto']
WHISPER_MODEL   = 'large-v3' #@param ['large-v3','medium','small']
CONF_MIN        = 0.50       #@param {type:"number"}
SEMITONE_CHANGE = 0.8        #@param {type:"number"}
MIN_NOTE_MS     = 100        #@param {type:"integer"}

# CONF_MIN      below this CREPE isn't tracking a real pitch
# SEMITONE_CHANGE  pitch movement that starts a new note
# MIN_NOTE_MS   shorter than this is a blip, not a note
print('ready')

In [ ]:
#@title 3b — Pitch tracker (crepe, with automatic torchcrepe fallback)
import numpy as np

_BACKEND = None

def _init_pitch():
    """Picks a working pitch tracker once. crepe is preferred - it is what
    the app itself uses - but it predates Keras 3 and can fail to build its
    model on current Colab TensorFlow. torchcrepe is the same algorithm in
    PyTorch and installs cleanly."""
    global _BACKEND
    if _BACKEND:
        return _BACKEND
    try:
        import crepe
        # Must actually run: crepe builds its Keras model on first predict,
        # which is where a version mismatch appears, not at import.
        crepe.predict(np.zeros(16000, np.float32), 16000,
                      model_capacity='tiny', verbose=0)
        _BACKEND = 'crepe'
    except Exception as e:
        print(f'crepe unusable ({type(e).__name__}: {e}) -> falling back')
        import sys, subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchcrepe'],
                       check=True)
        _BACKEND = 'torchcrepe'
    print('pitch backend:', _BACKEND)
    return _BACKEND


def crepe_predict(audio, sr):
    """-> (times, f0_hz, confidence) on a 10 ms grid, whichever backend works.
    Downstream code needs only those three, so the choice is invisible."""
    if _init_pitch() == 'crepe':
        import crepe
        t, f0, conf, _ = crepe.predict(audio, sr, viterbi=True,
                                       step_size=10, verbose=1)
        return t, f0, conf

    import torch, torchcrepe
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    hop = int(sr / 100)
    x = torch.tensor(audio, dtype=torch.float32, device=dev)[None]
    f0, periodicity = torchcrepe.predict(
        x, sr, hop, fmin=50.0, fmax=1100.0, model='full',
        batch_size=512, device=dev, return_periodicity=True)
    f0 = f0.squeeze().cpu().numpy()
    conf = periodicity.squeeze().cpu().numpy()
    return np.arange(len(f0)) * (hop / sr), f0, conf


crepe_predict(np.zeros(16000, np.float32), 16000)
print('crepe_predict ready')


In [ ]:
#@title 4 — The pipeline
import os, re, json, glob, subprocess
import numpy as np, librosa, whisper

NAMES = ['C','C\u266f','D','D\u266f','E','F','F\u266f','G','G\u266f','A','A\u266f','B']

_asr = None
def asr_model():
    """Loaded once and reused — large-v3 is a multi-GB download."""
    global _asr
    if _asr is None:
        print(f'loading whisper {WHISPER_MODEL} (first time downloads it)...')
        _asr = whisper.load_model(WHISPER_MODEL)
    return _asr


def isolate_vocal(mp3):
    """Demucs writes to separated/<model>/<stem>/vocals.wav; the model folder
    name changes between versions, so the file is located by glob rather
    than by assuming a path."""
    subprocess.run(['demucs', '--two-stems=vocals', '-o', '/content/sep', mp3],
                   check=True)
    stem = os.path.splitext(os.path.basename(mp3))[0]
    hits = glob.glob(f'/content/sep/*/{glob.escape(stem)}/vocals.wav')
    if not hits:
        raise FileNotFoundError('Demucs produced no vocal stem')
    return hits[0]


def notes_from(vocal_wav):
    """F0 contour -> discrete notes. A note ends when the pitch moves more
    than SEMITONE_CHANGE from the note's running median, or when the voice
    drops out."""
    audio, sr = librosa.load(vocal_wav, sr=16000, mono=True)   # CREPE wants 16 kHz
    t, f0, conf = crepe_predict(audio, sr)

    midi = np.where(f0 > 0, 69 + 12*np.log2(np.maximum(f0, 1e-6)/440.0), np.nan)
    voiced = conf >= CONF_MIN

    out, start, buf = [], None, []
    for i in range(len(t)):
        if voiced[i] and np.isfinite(midi[i]):
            if start is None:
                start, buf = t[i], [midi[i]]
            elif abs(midi[i] - np.median(buf)) > SEMITONE_CHANGE:
                out.append((start, t[i], float(np.median(buf))))
                start, buf = t[i], [midi[i]]
            else:
                buf.append(midi[i])
        elif start is not None:
            out.append((start, t[i], float(np.median(buf))))
            start, buf = None, []
    if start is not None:
        out.append((start, t[-1], float(np.median(buf))))

    return [n for n in out if (n[1]-n[0])*1000 >= MIN_NOTE_MS]


def words_from(vocal_wav):
    """Word timings, taken from the ISOLATED vocal rather than the mix —
    the backing track is exactly what makes transcription hard."""
    kw = {'word_timestamps': True}
    if LANGUAGE != 'auto':
        kw['language'] = LANGUAGE
    res = asr_model().transcribe(vocal_wav, **kw)
    return [{'w': w['word'].strip(), 's': w['start'], 'e': w['end']}
            for seg in res['segments'] for w in seg.get('words', [])
            if w['word'].strip()]


def build(mp3):
    print(f'\n=== {mp3} ===')
    vocal = isolate_vocal(mp3)
    notes = notes_from(vocal)
    print(f'  {len(notes)} notes')
    words = words_from(vocal)
    print(f'  {len(words)} words')

    def word_at(s, e):
        best, best_ov = '', 0.0
        for w in words:
            ov = min(e, w['e']) - max(s, w['s'])
            if ov > best_ov:
                best_ov, best = ov, w['w']
        return best

    data = [{'syllable': word_at(s, e),
             'start': round(float(s), 3),
             'end': round(float(e), 3),
             'note': f'{NAMES[int(round(m)) % 12]}{int(round(m))//12 - 1}'}
            for s, e, m in notes]

    title = TITLES.get(mp3, os.path.splitext(os.path.basename(mp3))[0])
    # Filename convention the app resolves by: title with underscores.
    safe = re.sub(r'[^A-Za-z0-9]+', '_', title).strip('_')
    out = f'/content/{safe}.json'
    json.dump({'data': data}, open(out, 'w', encoding='utf-8'),
              ensure_ascii=False, indent=1)
    print(f'  -> {out}  ({len(data)} entries)')
    return out, data


results = {}
for p in paths:
    try:
        results[p] = build(p)
    except Exception as exc:
        print(f'  FAILED: {type(exc).__name__}: {exc}')

In [ ]:
#@title 5 — Look at what came out
for src, (path, data) in results.items():
    print(f'\n=== {src} — {len(data)} entries ===')
    for d in data[:12]:
        print(f"  {d['start']:7.3f}-{d['end']:7.3f}  {d['note']:<4}  {d['syllable']}")
    named = sum(1 for d in data if d['syllable'])
    print(f"  entries with a word attached: {named}/{len(data)} "
          f"({100*named/max(len(data),1):.0f}%)")
    span = data[-1]['end'] - data[0]['start'] if data else 0
    print(f'  covers {span:.1f}s of the song')

## 6 — Validate before trusting it

Generated ground truth decides what the app tells singers they got wrong.
Check it against the one file that was authored by hand.

Run the pipeline on **Nasa Iyo Na Ang Lahat**, upload the existing
`Nasa_Iyo_Na_Ang_Lahat.json` from `assets/ground_truth/`, and compare.

In [ ]:
#@title 6 — Agreement with the hand-authored file
ref_up = files.upload()            # pick Nasa_Iyo_Na_Ang_Lahat.json
ref = json.load(open(list(ref_up)[0], encoding='utf-8'))['data']
gen = list(results.values())[0][1]  # the song you just generated

def at(entries, when):
    for e in entries:
        if e['start'] <= when <= e['end']:
            return e
    return None

exact = chroma = covered = 0
for r in ref:
    g = at(gen, (r['start'] + r['end']) / 2)
    if not g:
        continue
    covered += 1
    exact += g['note'] == r['note']
    # Same pitch class, wrong octave — the classic pitch-tracker error, worth
    # separating out because it has a different cause than being plain wrong.
    chroma += re.sub(r'\d', '', g['note']) == re.sub(r'\d', '', r['note'])

print(f'reference entries      : {len(ref)}')
print(f'covered by generated   : {covered} ({100*covered/len(ref):.0f}%)')
print(f'exact note match       : {exact}/{covered} = {100*exact/max(covered,1):.1f}%')
print(f'same pitch class       : {chroma}/{covered} = {100*chroma/max(covered,1):.1f}%')
print()
print('>=80% exact : good enough to generate the rest')
print('50-80%      : usable only if you say so in the writeup')
print('<50%        : singers would be scored against wrong notes - do not ship')

In [ ]:
#@title 7 — Download the JSON(s)
for path, _ in results.values():
    files.download(path)

print('\nDrop these into assets/ground_truth/ in the repo.')
print('The filename must match the song title with underscores, e.g.')
print('   "Nasa Iyo Na Ang Lahat"  ->  Nasa_Iyo_Na_Ang_Lahat.json')
print('An admin can also upload one from the Admin page at runtime.')